# 04 — Evaluation Final Visualization

**Source**: `eval/results.csv` (output dari `app.evaluation.runner`)

**Untuk laporan akhir**: charts di sini bisa di-save dan di-embed ke PDF report.

## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

plt.style.use('seaborn-v0_8-whitegrid')

## Load Results

In [ ]:
df = pd.read_csv('../eval/results.csv')
print(f'Total rows: {len(df)}')
print(f'Models: {df["model"].unique().tolist()}')
print(f'Queries: {df["query_id"].nunique()}')
df.head()

## Aggregate Metrics per Model

In [ ]:
agg = df.groupby('model').agg(
    p_at_5=('p_at_5', 'mean'),
    p_at_10=('p_at_10', 'mean'),
    map=('ap', 'mean'),
    ndcg_at_10=('ndcg_at_10', 'mean'),
    mrr=('rr', 'mean'),
).sort_values('map', ascending=False)
agg

## Chart 1: Bar Chart MAP per Model

In [ ]:
model_order = agg.index.tolist()
colors = {'bm25': '#2563eb', 'hybrid': '#7c3aed', 'tfidf': '#0891b2', 'indobert': '#dc2626'}

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(model_order, agg['map'], color=[colors.get(m, '#888') for m in model_order])
ax.set_ylabel('MAP')
ax.set_title(f'Mean Average Precision per IR Model (n={df["query_id"].nunique()} queries, corpus real)')
ax.set_ylim(0, max(agg['map']) * 1.15)
for bar, val in zip(bars, agg['map']):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.005, f'{val:.4f}',
            ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('charts/04_map_bars.png', dpi=100, bbox_inches='tight')
plt.show()

## Chart 2: All Metrics Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
metrics = ['p_at_5', 'p_at_10', 'map', 'ndcg_at_10', 'mrr']
x = np.arange(len(metrics))
width = 0.2

for i, model in enumerate(model_order):
    vals = [agg.loc[model, m] for m in metrics]
    ax.bar(x + i * width, vals, width, label=model, color=colors.get(model, '#888'))

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(['P@5', 'P@10', 'MAP', 'NDCG@10', 'MRR'])
ax.set_ylabel('Score')
ax.set_title('All Metrics — 4 IR Models Compared')
ax.legend()
plt.tight_layout()
plt.savefig('charts/04_all_metrics.png', dpi=100, bbox_inches='tight')
plt.show()

## Chart 3: Per-Query Heatmap (AP)

In [ ]:
pivot = df.pivot(index='query_id', columns='model', values='ap')
pivot = pivot[model_order]  # reorder columns

fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlGnBu', vmin=0, vmax=1)
ax.set_xticks(range(len(model_order)))
ax.set_xticklabels(model_order)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_title('Average Precision per Query × Model')
for i in range(len(pivot.index)):
    for j in range(len(model_order)):
        val = pivot.values[i, j]
        color = 'white' if val > 0.5 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', color=color, fontsize=8)
fig.colorbar(im, ax=ax, label='AP')
plt.tight_layout()
plt.savefig('charts/04_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()

## Chart 4: Query Difficulty (Mean AP across all models)

In [ ]:
query_difficulty = df.groupby('query_id')['ap'].mean().sort_values()
query_text = df.groupby('query_id')['query'].first()

fig, ax = plt.subplots(figsize=(10, 6))
colors_difficulty = ['#dc2626' if v < 0.1 else '#f59e0b' if v < 0.3 else '#10b981' for v in query_difficulty]
ax.barh(query_difficulty.index, query_difficulty.values, color=colors_difficulty)
ax.set_xlabel('Mean AP across all 4 models')
ax.set_title('Query Difficulty (sorted ascending)')
ax.axvline(0.1, color='red', linestyle='--', alpha=0.5, label='Hard (<0.1)')
ax.axvline(0.3, color='orange', linestyle='--', alpha=0.5, label='Medium (<0.3)')
ax.legend()
plt.tight_layout()
plt.savefig('charts/04_difficulty.png', dpi=100, bbox_inches='tight')
plt.show()

print('\nHARD queries (mean AP < 0.1):')
for qid in query_difficulty[query_difficulty < 0.1].index:
    print(f'  {qid}: {query_text[qid]!r}')

## Statistical Significance Summary

In [ ]:
from scipy import stats

print('Pairwise Wilcoxon signed-rank on AP per query (alpha=0.05):')
print('=' * 70)
models = list(model_order)
for i, ma in enumerate(models):
    for mb in models[i+1:]:
        a_aps = df[df['model'] == ma].sort_values('query_id')['ap'].values
        b_aps = df[df['model'] == mb].sort_values('query_id')['ap'].values
        try:
            stat, p = stats.wilcoxon(a_aps, b_aps)
            sig = 'SIGNIFICANT' if p < 0.05 else 'not sig'
            print(f'  {ma:10s} vs {mb:10s} | stat={stat:6.2f} | p={p:.4f} | {sig}')
        except Exception as e:
            print(f'  {ma} vs {mb}: ERROR {e}')

## Final Insights (corpus real 227, 15 queries)

1. **BM25 winner** (MAP 0.319, P@5 0.613) — 4/6 pairs SIGNIFICANT (signifikan vs IndoBERT & Hybrid; selisih vs TF-IDF tidak signifikan).
2. **IndoBERT bukan buruk — pooling bias**: MAP standard 0.053 → pool-restricted 0.527. GT di-pool dari BM25 sehingga hasil semantic unik tidak ter-judge. Bandingkan `results.csv` (standard) vs `results_pool_restricted.csv`.
3. **Hybrid #2 di pool-restricted** (MAP 0.594, MRR 0.783) — kombinasi bernilai saat eval fair.
4. **Data real terse**: deskripsi pemilik median ~23 kata menurunkan absolute metric tapi otentik.
5. **Sample n=15**: cukup untuk Wilcoxon (4/6 significant); 30+ ideal untuk future work.

> **Penting**: untuk evaluasi fair model non-lexical, lihat juga pool-restricted (`scripts/eval_pool_restricted.py`).